# IndexTTS2 - 部署笔记本 (Colab/Kaggle)

**解决导入悖论**: 必须先克隆代码，才能使用deploy模块

In [ ]:
# ============================================
# Cell 1: 环境准备 (独立运行，不依赖deploy模块)
# ============================================
import os, sys, json, subprocess

# 环境检测
IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = os.path.exists('/kaggle/input') or os.environ.get('KAGGLE_KERNEL_RUN_TYPE') is not None
WORK_DIR = '/content' if IN_COLAB else '/kaggle/working' if IN_KAGGLE else '/tmp'
REPO_DIR = f"{WORK_DIR}/index-tts"

print(f"🔍 环境: {'Colab' if IN_COLAB else 'Kaggle' if IN_KAGGLE else 'Other'}")
print(f"📁 工作目录: {WORK_DIR}")

# 基础依赖（克隆代码前就需要）
print("📦 安装基础依赖...")
subprocess.run(["pip", "install", "-q", "torch==2.8.0", "torchaudio==2.8.0", "--index-url", "https://download.pytorch.org/whl/cu121"], check=True)

basic_deps = ["pyngrok", "gradio==5.45.0", "fastapi", "uvicorn", "python-multipart", "huggingface-hub"]
for dep in basic_deps:
    subprocess.run(["pip", "install", "-q", dep], check=True)

# 克隆代码仓库
print("📥 克隆代码...")
if os.path.exists(REPO_DIR):
    subprocess.run(["rm", "-rf", REPO_DIR], check=True)
subprocess.run(["git", "clone", "-b", "dev", "https://github.com/infinite-gaming-studio/index-tts.git", REPO_DIR], check=True)

# 保存配置供后续使用
config = {"work_dir": WORK_DIR, "repo_dir": REPO_DIR, "in_colab": IN_COLAB, "in_kaggle": IN_KAGGLE}
json.dump(config, open("/tmp/notebook_config.json", "w"))

# 添加项目路径
sys.path.insert(0, REPO_DIR)
print(f"✅ 代码克隆完成! 路径: {REPO_DIR}")
print("👉 现在可以运行Cell 2使用deploy模块")

In [ ]:
# ============================================
# Cell 2: 完整安装 + 模型下载
# 注意: 必须在Cell 1运行成功后才能执行此Cell
# ============================================

# 此时deploy模块已可用
from deploy.utils import DependencyInstaller, ModelDownloader, check_model_exists

# 安装项目完整依赖
print("📦 安装完整依赖...")
DependencyInstaller.install_deps()

# 安装项目本身
subprocess.run(["pip", "install", "-q", "-e", REPO_DIR], check=True)

# 检查并下载模型
CHECKPOINT_DIR = f"{REPO_DIR}/checkpoints"
if not check_model_exists(CHECKPOINT_DIR):
    print("🤖 下载模型 (约3-8GB)...")
    ModelDownloader.download(CHECKPOINT_DIR, source="huggingface")
else:
    print("✅ 模型已存在")

print("✅ 全部准备完成! 可以启动服务了")

In [ ]:
# ============================================
# Cell 3: 启动服务
# ============================================
from deploy.launcher import quick_start

# 配置参数
PORT = 8000
MODE = "both"  # 选项: "api" | "webui" | "both"
NGROK_TOKEN = None  # 替换为你的token获取公网访问

# 启动服务
launcher = quick_start(port=PORT, mode=MODE, ngrok_token=NGROK_TOKEN)

In [ ]:
# ============================================
# Cell 4: 服务管理 (可选)
# ============================================

# 查看最新日志
launcher.logs(lines=30)

# 检查服务状态
# launcher.status()

# 停止服务
# launcher.stop()